# 16. Audit Dataset
**Penelitian:** Deteksi anemia dari citra pucat (telapak tangan, kuku, konjungtiva) dengan data pooling multi-situs.

**Tujuan:** memeriksa kualitas berkas citra: kategori nama file, citra identik (MD5), pasangan citra identik berlabel berbeda, dan struktur salinan bernomor. Notebook ini tidak mengubah atau menghapus file.

**Masukan:** Dataset (Mendeley Data): Palm doi:10.17632/ccr8cm22vz.1, Nail doi:10.17632/2xx4j3kjg2.1, Conjunctiva doi:10.17632/nt7r8hv2pz.1. Letakkan dalam folder `Dataset_Multisite/{Palm,Nail,Conjunctiva}`.

**Keluaran (folder `hasil_revisi`):** `audit_kategori_file.csv`, `file_terpakai_md5.csv`.

**Kaitan dengan naskah:** Bagian 2.1 (audit dataset) dan Tabel 1.

**Cara menjalankan:** Colab, Runtime > Run all. CPU sudah cukup; GPU tidak diperlukan. Seed acak = 42 dan urutan file diurutkan (`sorted`) sehingga hasil dapat direproduksi pada versi pustaka yang dicetak di sel kedua.

In [ ]:
!pip install lightgbm -q

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

Mounted at /content/drive


In [ ]:
# ===== FUNGSI BERSAMA (identik di keempat notebook) =====
import os, re, hashlib, warnings
import numpy as np, pandas as pd, cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

# ---------- KONFIGURASI (ubah hanya bagian ini) ----------
SEED = 42
BASE = '/content/drive/MyDrive/Dataset_Multisite'   # folder berisi subfolder Palm/, Nail/, Conjunctiva/
OUT  = '/content/drive/MyDrive/hasil_revisi'          # folder keluaran (CSV)
SITES = {'Palm': 0, 'Nail': 1, 'Conjunctiva': 2}
# Filter piksel memeriksa kanal L, R, dan G. Kanal B ikut difilter jika True
# (Appiahene et al. 2023 menuliskan kondisi min < R=G=B < max).
FILTER_INCLUDE_B = False
os.makedirs(OUT, exist_ok=True)

# ---------- DAFTAR FILE ----------
def parse_label(name):
    """Label dari awalan nama file (Anemic / Non-Anemic); nama yang mengandung tanda kurung (salinan bernomor) dan awalan tidak dikenali dilewati."""
    if '(' in name: return None
    c = name.lower().replace(' ', '')
    if c.startswith('non-anemic'): return 0
    if c.startswith('anemic'): return 1
    return None

def list_used_files():
    rows = []
    for site, sid in SITES.items():
        d = os.path.join(BASE, site)
        for n in sorted(os.listdir(d)):          # sorted -> urutan deterministik
            lab = parse_label(n)
            if lab is not None:
                rows.append(dict(site=site, site_id=sid, file=n, path=os.path.join(d, n), label=lab))
    return pd.DataFrame(rows)

def md5(path):
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()

# ---------- EKSTRAKSI FITUR ----------
def feat_original(img, lo=50, hi=200):
    """Fitur [mean a*, mean b*, mean G] dengan filter piksel. Return (fitur, mask_kosong)."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab); R, G, B = cv2.split(rgb)
    m = (L > lo) & (L < hi) & (R > lo) & (R < hi) & (G > lo) & (G < hi)
    if FILTER_INCLUDE_B: m &= (B > lo) & (B < hi)
    if m.any(): return [a[m].mean(), b[m].mean(), G[m].mean()], False
    return [0.0, 0.0, 0.0], True

def feat_otsu(img):
    """Fitur dari piksel foreground Otsu (grayscale). Tanpa filter L/R/G. Return (fitur, mask_kosong)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mk = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = mk > 0
    if not m.any(): return [0.0, 0.0, 0.0], True
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    _, a, b = cv2.split(lab); G = rgb[:, :, 1]
    return [a[m].mean(), b[m].mean(), G[m].mean()], False

# ---------- PEMBAGIAN DATA & EVALUASI ----------
def split_idx(y, scheme, seed=SEED):
    """Return (train, val, test) berupa indeks. Skema: '80:20', '80:10:10', '70:10:20'."""
    idx = np.arange(len(y))
    if scheme == '80:20':
        tr, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        return tr, None, te
    if scheme == '80:10:10':
        tr, tmp = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
        return tr, va, te
    if scheme == '70:10:20':
        trva, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        tr, va = train_test_split(trva, test_size=0.125, stratify=y[trva], random_state=seed)
        return tr, va, te
    raise ValueError(scheme)

def metrics_from(yte, pred, proba):
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(yte), 'TP+TN+FP+FN harus sama dengan jumlah data uji'
    two = len(set(yte)) == 2
    return dict(n_test=len(yte), n_test_anemic=int((yte == 1).sum()), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                acc=100*(tp+tn)/len(yte),
                sens=100*tp/(tp+fn) if tp+fn else np.nan,
                spec=100*tn/(tn+fp) if tn+fp else np.nan,
                f1=100*f1_score(yte, pred, zero_division=0),
                auc=100*roc_auc_score(yte, proba) if two else np.nan,
                baseline_mayoritas=100*max((yte == 1).mean(), (yte == 0).mean()))

def fit_lgbm(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    m = LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1).fit(sc.transform(Xtr), ytr)
    return sc, m

def predict_lgbm(sc, m, X):
    proba = m.predict_proba(sc.transform(X))[:, 1]
    return (proba >= 0.5).astype(int), proba

def bersihkan(df):
    """Buang salinan identik (MD5; simpan satu) dan KEDUA file bila citra identik berlabel berbeda."""
    df = df.copy(); df['md5'] = df['path'].map(md5)
    konflik = df.groupby('md5').label.transform('nunique') > 1
    salinan = df.duplicated('md5', keep='first')
    return df[~(konflik | salinan)].reset_index(drop=True)

def cek_kebocoran(md5s, tr, va, te, wajib=True):
    """Pemeriksaan kebocoran: indeks latih/validasi/uji harus saling lepas dan tidak ada citra identik (MD5) lintas partisi.
    Return jumlah citra identik lintas partisi, mis. {'latih-uji': 0}. wajib=True -> hentikan program bila ada pelanggaran."""
    parts = {'latih': set(tr), 'uji': set(te)}
    if va is not None: parts['val'] = set(va)
    nm = sorted(parts)
    for i, a in enumerate(nm):
        for b in nm[i + 1:]:
            assert not (parts[a] & parts[b]), f'Indeks tumpang tindih: {a}-{b}'
    m = {k: set(np.asarray(md5s)[sorted(v)]) for k, v in parts.items()}
    bocor = {f'{a}-{b}': len(m[a] & m[b]) for i, a in enumerate(nm) for b in nm[i + 1:]}
    if wajib: assert all(v == 0 for v in bocor.values()), f'Citra identik lintas partisi: {bocor}'
    return bocor

In [ ]:
import sys, numpy, pandas, cv2, sklearn, lightgbm
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| OpenCV', cv2.__version__, '| scikit-learn', sklearn.__version__, '| LightGBM', lightgbm.__version__, '| SEED', SEED)

Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | OpenCV 4.14.0 | scikit-learn 1.6.1 | LightGBM 4.6.0 | SEED 42


In [ ]:
# ===== AUDIT DATASET: kategori nama file, duplikat (MD5), salinan bernomor =====
rows = []
for site in SITES:
    names = sorted(os.listdir(os.path.join(BASE, site)))
    groups = {}
    for n in names:
        c = n.lower().replace(' ', '')
        lab = 'non-anemic' if c.startswith('non-anemic') else ('anemic' if c.startswith('anemic') else 'tidak_dikenali')
        groups.setdefault(('berkurung' if '(' in n else 'tanpa_kurung', lab), []).append(n)
    for (kur, lab), lst in sorted(groups.items()):
        rows.append(dict(situs=site, nama=kur, label=lab, jumlah=len(lst), contoh=' | '.join(lst[:4])))
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
audit.to_csv(os.path.join(OUT, 'audit_kategori_file.csv'), index=False)

      situs         nama          label  jumlah                                                                                                            contoh
       Palm    berkurung         anemic    2258                              Anemic-260 (10).png | Anemic-260 (11).png | Anemic-260 (12).png | Anemic-260 (2).png
       Palm    berkurung     non-anemic    1476             Non-AnemicP-001 (2).png | Non-AnemicP-001 (3).png | Non-AnemicP-001 (4).png | Non-AnemicP-001 (5).png
       Palm tanpa_kurung         anemic     304                                                 Anemic-262.png | Anemic-264.png | Anemic-265.png | Anemic-266.png
       Palm tanpa_kurung     non-anemic     222                             Non-AnemicP-001.png | Non-AnemicP-002.png | Non-AnemicP-003.png | Non-AnemicP-004.png
       Nail    berkurung         anemic    2229                  Anemic-FN-001 (10).png | Anemic-FN-001 (11).png | Anemic-FN-001 (12).png | Anemic-FN-001 (2).png
       Nail    berkurung    

In [ ]:
# ===== File terpakai: duplikat identik (MD5), pola nama mencurigakan, dan pasangan salinan bernomor =====
df = list_used_files()
df['md5'] = df['path'].map(md5)
df['duplikat_ke_n'] = df.groupby('md5').cumcount()          # 0 = kemunculan pertama
ringkas = []
for site, g in df.groupby('site'):
    ringkas.append(dict(situs=site, total=len(g), anemic=int((g.label == 1).sum()), non_anemic=int((g.label == 0).sum()),
                        salinan_identik_MD5=int((g.duplikat_ke_n > 0).sum()),
                        unik=int((g.duplikat_ke_n == 0).sum()),
                        unik_anemic=int(((g.label == 1) & (g.duplikat_ke_n == 0)).sum()),
                        unik_non_anemic=int(((g.label == 0) & (g.duplikat_ke_n == 0)).sum()),
                        nama_akhiran_spasi_angka=int(g.file.str.contains(r'\s\d+\.[A-Za-z]+$').sum())))
ringkas = pd.DataFrame(ringkas)
print(ringkas.to_string(index=False))

konflik = df.groupby('md5').filter(lambda g: g.label.nunique() > 1)
print('\nCitra identik dengan label berbeda (harus 0):', len(konflik))
lintas = df.groupby('md5').filter(lambda g: g.site.nunique() > 1)
print('Citra identik lintas situs (harus 0):', len(lintas))

# Apakah file berkurung adalah salinan dari file yang dipakai, atau "yatim" (aslinya ikut terbuang)?
print('\nFile berkurung: punya pasangan (nama dasar ada di file terpakai) vs yatim')
for site in SITES:
    names = os.listdir(os.path.join(BASE, site))
    terpakai = set(df.loc[df.site == site, 'file'])
    brk = [n for n in names if '(' in n]
    base = lambda n: re.sub(r'\s*\(\d+\)', '', n)
    pasangan = sum(base(n) in terpakai for n in brk)
    print(f'  {site}: berkurung={len(brk)}, punya pasangan={pasangan}, yatim={len(brk)-pasangan}')

varian = df[df.file.str.contains(r'\d(FV|RA|FH|RO)\.', regex=True)]
print('\nFile terpakai bernama varian augmentasi (akhiran FV/RA/FH/RO):', len(varian), varian.file.tolist()[:10])

df.drop(columns=['path']).to_csv(os.path.join(OUT, 'file_terpakai_md5.csv'), index=False)
print('\nTersimpan di', OUT)

      situs  total  anemic  non_anemic  salinan_identik_MD5  unik  unik_anemic  unik_non_anemic  nama_akhiran_spasi_angka
Conjunctiva    547     322         225                   31   516          293              223                         0
       Nail    316     296          20                    1   315          295               20                         1
       Palm    526     304         222                    4   522          304              218                         0

Citra identik dengan label berbeda (harus 0): 2
Citra identik lintas situs (harus 0): 0

File berkurung: punya pasangan (nama dasar ada di file terpakai) vs yatim
  Palm: berkurung=3734, punya pasangan=3722, yatim=12
  Nail: berkurung=3708, punya pasangan=2323, yatim=1385
  Conjunctiva: berkurung=3722, punya pasangan=3706, yatim=16

File terpakai bernama varian augmentasi (akhiran FV/RA/FH/RO): 6 ['Anemic-268FV.png', 'Anemic-268RA.png', 'Anemic-FN-011FV.png', 'Anemic-FN-011RA.png', 'Anemic-001FV.png', 'Ane

In [ ]:
# ===== Struktur keluarga file: menguji dugaan 'nama dasar + salinan bernomor' (mis. 710 x 6 = 4.260 pada Palm) =====
for site in SITES:
    names = os.listdir(os.path.join(BASE, site)); nameset = set(names)
    fam = {}
    for n in names:
        fam.setdefault(re.sub(r'\s*\(\d+\)', '', n), []).append(n)
    ukuran = pd.Series({b: len(v) for b, v in fam.items()})
    tanpa_dasar = sum(b not in nameset for b in fam)
    print(f'{site}: {len(fam)} keluarga nama | sebaran jumlah file per keluarga: {ukuran.value_counts().sort_index().to_dict()}')
    print(f'   keluarga yang file dasarnya (tanpa kurung) TIDAK ada: {tanpa_dasar}')

Palm: 528 keluarga nama | sebaran jumlah file per keluarga: {1: 1, 6: 344, 11: 1, 12: 182}
   keluarga yang file dasarnya (tanpa kurung) TIDAK ada: 2
Nail: 556 keluarga nama | sebaran jumlah file per keluarga: {1: 33, 2: 6, 5: 1, 6: 337, 7: 3, 10: 1, 11: 2, 12: 163, 13: 1, 15: 2, 17: 4, 19: 1, 22: 1, 31: 1}
   keluarga yang file dasarnya (tanpa kurung) TIDAK ada: 0
Conjunctiva: 557 keluarga nama | sebaran jumlah file per keluarga: {1: 29, 2: 3, 3: 2, 4: 1, 5: 1, 6: 336, 7: 4, 12: 178, 13: 3}
   keluarga yang file dasarnya (tanpa kurung) TIDAK ada: 10


In [1]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/hasil_revisi'          # folder berisi CSV hasil
LABEL = ['Non-anemic', 'Anemic']

def cm_dari_baris(r):
    """Matriks [[TN, FP], [FN, TP]]: baris = kelas sebenarnya, kolom = prediksi."""
    cm = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]], dtype=int)
    n = r['n_test'] if 'n_test' in r.index else r['n']
    assert cm.sum() == n, f'TP+TN+FP+FN ({cm.sum()}) harus sama dengan jumlah data ({n})'
    return cm

def gambar_cm(cm, judul, ax):
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100      # persen per kelas sebenarnya
    ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]}\n({pct[i, j]:.1f}%)', ha='center', va='center', fontsize=9,
                    color='white' if pct[i, j] > 50 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(LABEL, fontsize=8); ax.set_yticks([0, 1]); ax.set_yticklabels(LABEL, fontsize=8)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya'); ax.set_title(judul, fontsize=8)

def grid_cm(df, judul, ncol=4, simpan=None, judul_utama=''):
    n = len(df); nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.4 * nrow), squeeze=False)
    for ax in axs.ravel(): ax.axis('off')
    for ax, (_, r) in zip(axs.ravel(), df.iterrows()):
        ax.axis('on'); gambar_cm(cm_dari_baris(r), judul(r), ax)
    fig.suptitle(judul_utama); plt.tight_layout()
    if simpan: plt.savefig(os.path.join(OUT, simpan), dpi=150)
    plt.show()